# Ridership Aggregation by Station and Time


## Build cleaned aggregated dataset


In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
drive.mount('/content/drive')


csvsFolderPath = '/content/drive/My Drive/CIS 4500 HW/CIS4500FinalProject/ProcessedDataCSVs'



Mounted at /content/drive


In [ ]:
def cleanDf(df):
  group_cols = ['transit_timestamp', 'transit_mode', 'station_complex_id','station_complex', 'borough', 'latitude', 'longitude', 'year', 'quarter']
  df_no_pay_agg = df.drop(columns=['payment_method', 'fare_class_category']).groupby(group_cols, as_index=False)[['ridership', 'transfers']].sum()

  group_cols = ['transit_timestamp', 'transit_mode', 'station_complex_id','station_complex', 'borough', 'year', 'quarter']
  df_no_pay_agg_no_coord = df_no_pay_agg.drop(columns=['latitude','longitude']).drop_duplicates().groupby(group_cols, as_index=False)[['ridership','transfers']].sum()

  #save this
  df_comp_id_useless_data = df_no_pay_agg_no_coord[['station_complex_id', 'transit_mode', 'station_complex', 'borough']].drop_duplicates()

  grpcols = ['transit_timestamp', 'station_complex_id', 'year', 'quarter']
  df_no_pay_agg_no_coord_no_useless = df_no_pay_agg_no_coord.drop(columns=['transit_mode', 'station_complex', 'borough']) #no dups (i checked)

  datetime_series = pd.to_datetime(df_no_pay_agg_no_coord_no_useless['transit_timestamp'])
  df_final_millis = df_no_pay_agg_no_coord_no_useless.copy()
  df_final_millis['transit_timestamp_millis'] = datetime_series.astype('int64') // 10**6
  df_final_millis.drop(columns=['transit_timestamp','year','quarter'], inplace=True)

  return df_final_millis, df_comp_id_useless_data

In [ ]:
import os
outputCSVPath = '/content/drive/My Drive/CIS 4500 HW/CIS4500FinalProject/UltraProcessedDataCSVsRECENT'

for year in range(2020, 2025):
  csv_file_path = os.path.join(csvsFolderPath, "ridership" + str(year) + "Dataframe.csv")
  df = pd.read_csv(csv_file_path)
  clean_df, df_useless_data = cleanDf(df)

  print(clean_df.shape)
  print(clean_df.head())
  print(df_useless_data.shape)
  print(df_useless_data.head())
  print("Saving to drive...")
  clean_df.to_csv(os.path.join(outputCSVPath, "ridership" + str(year) + "Dataframe.csv"), index=False)
  df_useless_data.to_csv(os.path.join(outputCSVPath, "ridership" + str(year) + "UselessDataDataframe.csv"), index=False)
  print("Saved to drive.")



(3390612, 4)
  station_complex_id  ridership  transfers  transit_timestamp_millis
0                501        110         19             1577840400000
1                502          5          2             1577840400000
2                  1        172          0             1577840400000
3                 10        400          0             1577840400000
4                100        103          9             1577840400000
(428, 4)
  station_complex_id           transit_mode             station_complex  \
0                501  staten_island_railway             St George (SIR)   
1                502  staten_island_railway         Tompkinsville (SIR)   
2                  1                 subway  Astoria-Ditmars Blvd (N,W)   
3                 10                 subway               49 St (N,R,W)   
4                100                 subway              Hewes St (M,J)   

         borough  
0  Staten Island  
1  Staten Island  
2         Queens  
3      Manhattan  
4       Brooklyn  

## Validation checks


In [ ]:
#@title station_complex_id is private key for many things
df[['transit_mode', 'station_complex_id', 'station_complex', 'borough']].drop_duplicates()

,transit_mode,station_complex_id,station_complex,borough
0,staten_island_railway,501,St George (SIR),Staten Island
1,subway,41,"7 Av (B,Q)",Brooklyn
2,subway,158,"86 St (C,B)",Manhattan
3,subway,187,Shepherd Av (C),Brooklyn
4,subway,636,"Jay St-MetroTech (A,C,F,R)",Brooklyn
...,...,...,...,...
2863,subway,202,"Beach 105 St (A,S)",Queens
3679,subway,253,Neptune Av (F),Brooklyn
5758,subway,446,Morris Park (5),Bronx
9915,subway,222,Roosevelt Island (F),Manhattan


In [ ]:
df[(df['station_complex_id'] == "501") & (df['transit_timestamp'] == "01/01/2024 12:00:00 AM")]

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,year,quarter
0,01/01/2024 12:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,metrocard,Metrocard - Other,7,1,40.64375,-74.07365,2024,1
117,01/01/2024 12:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,metrocard,Metrocard - Unlimited 7-Day,1,0,40.64375,-74.07365,2024,1
286,01/01/2024 12:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,metrocard,Metrocard - Fair Fare,4,1,40.64375,-74.07365,2024,1
324,01/01/2024 12:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,metrocard,Metrocard - Full Fare,7,1,40.64375,-74.07365,2024,1
1194,01/01/2024 12:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,metrocard,Metrocard - Seniors & Disability,5,1,40.64375,-74.07365,2024,1
1244,01/01/2024 12:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,omny,OMNY - Full Fare,12,6,40.64375,-74.07365,2024,1


In [ ]:
# Define the columns that define a unique record (excluding payment details)
group_cols = [
    'transit_timestamp', 'transit_mode', 'station_complex_id',
    'station_complex', 'borough', 'latitude', 'longitude', 'year', 'quarter'
]

df_no_pay = df.drop(columns=['payment_method', 'fare_class_category'])
df_no_pay_agg = df_no_pay.groupby(group_cols, as_index=False)[['ridership', 'transfers']].sum()

df_no_pay_agg[(df_no_pay_agg['station_complex_id'] == "501") & (df_no_pay_agg['transit_timestamp'] == "01/01/2024 12:00:00 AM")]

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,latitude,longitude,year,quarter,ridership,transfers
9793,01/01/2024 12:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,40.64375,-74.07365,2024,1,36,10


In [ ]:
#TODO: fix this
df_comp_id_coords = df_no_pay_agg[['station_complex_id', 'latitude', 'longitude']].drop_duplicates().copy()

group_cols = ['transit_timestamp', 'transit_mode', 'station_complex_id',
    'station_complex', 'borough', 'year', 'quarter']

df_no_pay_agg_no_coord = df_no_pay_agg.drop(columns=['latitude','longitude']).drop_duplicates().groupby(group_cols, as_index=False)[['ridership','transfers']].sum()
df_no_pay_agg_no_coord

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,year,quarter,ridership,transfers
0,01/01/2024 01:00:00 AM,staten_island_railway,501,St George (SIR),Staten Island,2024,1,92,18
1,01/01/2024 01:00:00 AM,staten_island_railway,502,Tompkinsville (SIR),Staten Island,2024,1,5,1
2,01/01/2024 01:00:00 AM,subway,1,"Astoria-Ditmars Blvd (N,W)",Queens,2024,1,103,0
3,01/01/2024 01:00:00 AM,subway,10,"49 St (N,R,W)",Manhattan,2024,1,300,0
4,01/01/2024 01:00:00 AM,subway,100,"Hewes St (M,J)",Brooklyn,2024,1,60,0
...,...,...,...,...,...,...,...,...,...
3682250,12/31/2024 12:00:00 PM,subway,97,"Myrtle Av (M,J,Z)",Brooklyn,2024,4,364,6
3682251,12/31/2024 12:00:00 PM,subway,98,"Flushing Av (M,J)",Brooklyn,2024,4,282,10
3682252,12/31/2024 12:00:00 PM,subway,99,"Lorimer St (M,J)",Brooklyn,2024,4,147,3
3682253,12/31/2024 12:00:00 PM,tram,TRAM1,RI Tramway (Manhattan),Manhattan,2024,4,439,132


In [ ]:
#TODO: fix this also
df_comp_id_useless_data = df_no_pay_agg_no_coord[['station_complex_id', 'transit_mode', 'station_complex', 'borough']].drop_duplicates()

grpcols = ['transit_timestamp', 'station_complex_id', 'year', 'quarter']
df_no_pay_agg_no_coord_no_useless = df_no_pay_agg_no_coord.drop(columns=['transit_mode', 'station_complex', 'borough']) #no dups (i checked)
df_no_pay_agg_no_coord_no_useless

,transit_timestamp,station_complex_id,year,quarter,ridership,transfers
0,01/01/2024 01:00:00 AM,501,2024,1,92,18
1,01/01/2024 01:00:00 AM,502,2024,1,5,1
2,01/01/2024 01:00:00 AM,1,2024,1,103,0
3,01/01/2024 01:00:00 AM,10,2024,1,300,0
4,01/01/2024 01:00:00 AM,100,2024,1,60,0
...,...,...,...,...,...,...
3682250,12/31/2024 12:00:00 PM,97,2024,4,364,6
3682251,12/31/2024 12:00:00 PM,98,2024,4,282,10
3682252,12/31/2024 12:00:00 PM,99,2024,4,147,3
3682253,12/31/2024 12:00:00 PM,TRAM1,2024,4,439,132


In [ ]:
print(df_comp_id_useless_data.dtypes)

station_complex_id    object
transit_mode          object
station_complex       object
borough               object
dtype: object


In [ ]:
datetime_series = pd.to_datetime(df_no_pay_agg_no_coord_no_useless['transit_timestamp'])

df_final_millis = df_no_pay_agg_no_coord_no_useless.copy()
df_final_millis['transit_timestamp_millis'] = datetime_series.astype('int64') // 10**6
df_final_millis.drop(columns=['transit_timestamp'], inplace=True)

print(df_final_millis.dtypes)

station_complex_id          object
year                         int64
quarter                      int64
ridership                    int64
transfers                    int64
transit_timestamp_millis     int64
dtype: object


## Save outputs
